In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os

project_root = Path.cwd().resolve()
# Notebook lives at models/wnba/ — climb to repo root if needed
if not (project_root / "data").exists():
    project_root = project_root.parent.parent

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)
os.chdir(project_root_str)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)


In [ ]:
from src.pipeline.features.context_features import ContextFeatureEngineer

seasons = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
frames = []
for yr in seasons:
    path = f"data/processed/wnba_{yr}_Regular_Season_training_data.parquet"
    season_df = pd.read_parquet(path)
    season_df = ContextFeatureEngineer(league="wnba").enrich(season_df)
    frames.append(season_df)

df = pd.concat(frames, ignore_index=True)
df.sample(10)


In [ ]:
print(f"Total number of duplicates: {df.duplicated(subset=['game_id', 'player_id']).sum()}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df[['minutes', 'pts_per_min', 'ast_per_min', 'reb_per_min']].describe()


In [ ]:
# ── Prop-specific config (WNBA APM) ───────────────────────────────────────────
APM_FEATURES = [
    "base_ast_per_min_ewm_hl10",
    "adv_ast_pct_ewm_hl10",
    "adv_poss_ewm_hl10",
    "adv_usg_pct_ewm_hl10",
    "team_ast_per_min_rank_l10",
]

HOLDOUT_SEASON = "2026"
ID_COLS = ["game_id", "player_id", "season_year", "player_name", "game_date"]
TARGET_COL = "ast_per_min"
ROLE_COL = "starting"

# Frozen naive baselines for the APM hypothesis test (leakage-safe).
NAIVE_PRIMARY = "base_ast_per_min_season_avg"
NAIVE_SECONDARY = "base_ast_per_min_lag1"
NAIVE_COLS = [NAIVE_PRIMARY, NAIVE_SECONDARY]
ALPHA = 0.05

QUANTILES = [0.10, 0.50, 0.90]

XGB_PARAMS = dict(
    objective="reg:quantileerror",
    n_estimators=1553,
    max_depth=4,
    learning_rate=0.016209908567987385,
    subsample=0.6707418178831205,
    colsample_bytree=0.5074990705838847,
    reg_alpha=0.33354531117582426,
    reg_lambda=0.06707791013177704,
    min_child_weight=6,
    n_jobs=-1,
    random_state=42,
    early_stopping_rounds=50,
)

APM_TIERS = {
    "<0.10 ast/min": lambda a: a < 0.10,
    "0.10-0.20 ast/min": lambda a: (a >= 0.10) & (a < 0.20),
    "0.20-0.35 ast/min": lambda a: (a >= 0.20) & (a < 0.35),
    "0.35+ ast/min": lambda a: a >= 0.35,
}

ARTIFACT_STEM = "apm_wnba_model"

# ── Shared train/eval helpers ─────────────────────────────────────────────────
from models.shared.splits import prepare_splits
from models.shared.train import run_timeseries_cv, run_walk_forward, evaluate_holdout
from models.shared.baselines import run_naive_comparison, evaluate_holdout_vs_naive
from models.shared.analysis import run_feature_ablation, analyze_correlations
from models.shared.artifacts import save_model_bundle, load_model_bundle, predict_quantiles
from models.shared.metrics import pinball_50

print(f"Number of features: {len(APM_FEATURES)}")
APM_FEATURES


In [ ]:
df = df[(df['minutes'] >= 10) | (df['starting'] == 1)]
print(f"After filtering minutes: {df[APM_FEATURES].shape[0]} rows")


In [ ]:
splits = prepare_splits(
    df,
    holdout_season=HOLDOUT_SEASON,
    features=APM_FEATURES,
    target_col=TARGET_COL,
    naive_primary=NAIVE_PRIMARY,
    naive_secondary=NAIVE_SECONDARY,
    id_cols=ID_COLS,
    role_col=ROLE_COL,
    extra_keep=["minutes"],
)
ppm_df = splits["ppm_df"]
ppm_holdout = splits["ppm_holdout"]
X = splits["X"]
y = splits["y"]


In [ ]:
tscv_results = run_timeseries_cv(
    X, y, ppm_df,
    xgb_params=XGB_PARAMS,
    role_col=ROLE_COL,
    tiers=APM_TIERS,
    quantiles=QUANTILES,
)
wf = run_walk_forward(
    X, y, ppm_df,
    xgb_params=XGB_PARAMS,
    role_col=ROLE_COL,
    tiers=APM_TIERS,
    quantiles=QUANTILES,
)

wf_results = wf["wf_results"]
models_last = wf["models_last"]
preds_last = wf["preds_last"]
X_val_last = wf["X_val_last"]
y_val_last = wf["y_val_last"]
starting_last = wf["starting_last"]
last_fold = wf["last_fold"]


### Naive baseline — APM hypothesis test

**H0:** The APM model does not predict significantly better than a naive baseline.  
**H1:** The APM model predicts significantly better than naive.

**Frozen primary naive:** `base_ast_per_min_season_avg` (season-to-date expanding mean, shift-then-expand — no same-game leakage).  
**Secondary (reported only):** `base_ast_per_min_lag1` (last-game ast/min).

**Decision rule:** on the same player-games, reject H0 if model MAE < naive MAE **and** one-sided Wilcoxon signed-rank on `|y − ŷ|` has p < 0.05 (model errors stochastically smaller).


In [ ]:
naive = run_naive_comparison(
    wf_results,
    ppm_df,
    target_col=TARGET_COL,
    naive_primary=NAIVE_PRIMARY,
    naive_secondary=NAIVE_SECONDARY,
    y_val_last=y_val_last,
    preds_last=preds_last,
    last_fold=last_fold,
    alpha=ALPHA,
)


### Residual Visuals


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error

y_true = y_val_last.values
pred_low = preds_last["q_0.10"]
pred_mid = preds_last["q_0.50"]
pred_high = preds_last["q_0.90"]
preds_plot = {"Q10": pred_low, "Q50": pred_mid, "Q90": pred_high}

fig, axes = plt.subplots(1, 3, figsize=(14, 6), sharex=True, sharey=True)
scatter_metrics = {}

for ax, (name, y_pred) in zip(axes, preds_plot.items()):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    scatter_metrics[name] = {"R2": r2, "MAE": mae}
    ax.scatter(y_true, y_pred, s=12, alpha=0.6, edgecolors="none")
    lims = [0, max(y_true.max(), y_pred.max()) * 1.02]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_title(f"{name}  |  R²={r2:.3f}  MAE={mae:.4f}")
    ax.set_xlabel("Actual AST/MIN")
    ax.set_ylabel("Predicted AST/MIN")
    ax.set_aspect("equal", adjustable="box")
    ax.text(
        0.04, 0.96,
        f"R² = {r2:.3f}\nMAE = {mae:.4f}",
        transform=ax.transAxes, ha="left", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85, edgecolor="0.7"),
    )

plt.suptitle(f"Walk-Forward last fold — {last_fold['fold']}", y=1.02)
plt.tight_layout()
plt.show()

print("Per-quantile fit on last validation fold:")
for name, m in scatter_metrics.items():
    print(f"  {name}: R²={m['R2']:+.3f}  MAE={m['MAE']:.4f}")

print("\nCoverage per quantile (should match quantile value):")
for name, y_pred in preds_plot.items():
    coverage = (y_true <= y_pred).mean()
    q_target = {"Q10": 0.10, "Q50": 0.50, "Q90": 0.90}[name]
    delta = coverage - q_target
    flag = "⚠" if abs(delta) > 0.05 else "✓"
    print(f"  {name}: {coverage:.1%}  (target {q_target:.0%}, delta {delta:+.1%})  {flag}")

interval_width = pred_high - pred_low
print(f"\n80% interval width (Q10→Q90):")
print(f"  mean   : {interval_width.mean():.4f} ast/min")
print(f"  median : {np.median(interval_width):.4f} ast/min")
print(f"  std    : {interval_width.std():.4f} ast/min")

residuals = y_true - pred_mid
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(pred_mid, residuals, s=12, alpha=0.5, edgecolors="none")
axes[0].axhline(0, color="k", lw=1, linestyle="--")
axes[0].set_xlabel("Predicted AST/MIN (Q50)")
axes[0].set_ylabel("Residual (actual − predicted)")
axes[0].set_title("Residuals vs Predicted")
axes[1].hist(residuals, bins=40, edgecolor="none")
axes[1].axvline(0, color="k", lw=1, linestyle="--")
axes[1].set_xlabel("Residual")
axes[1].set_title("Residual Distribution")
plt.tight_layout()
plt.show()

print(f"\nResidual summary:")
print(f"  mean bias : {residuals.mean():+.4f}  (close to 0 = unbiased)")
print(f"  std       : {residuals.std():.4f}")

starting_last = ppm_df.loc[X_val_last.index, "starting"].values
print("\nResiduals by role:")
for role, mask in [("Starters", starting_last == 1), ("Bench", starting_last == 0)]:
    if mask.sum() == 0:
        continue
    r = residuals[mask]
    print(f"  {role:10s} | n={mask.sum():5d} | bias={r.mean():+.4f} | std={r.std():.4f}")

print("\nResiduals by APM tier:")
for tier, fn in APM_TIERS.items():
    mask = fn(y_true)
    if mask.sum() == 0:
        continue
    r = residuals[mask]
    print(f"  {tier:18s} | n={mask.sum():5d} | bias={r.mean():+.4f} | std={r.std():.4f}")


In [ ]:
import shap

for q_key, model in models_last.items():
    explainer = shap.TreeExplainer(model, model_output="raw")
    shap_values = explainer(X_val_last)
    print(f"\n── {q_key} ──")
    shap.plots.beeswarm(shap_values, max_display=30, show=False)
    plt.title(f"SHAP beeswarm — {q_key}")
    plt.tight_layout()
    plt.show()


In [ ]:
model = models_last["q_0.50"]
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_val_last)

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
feat_imp = pd.Series(mean_abs_shap, index=APM_FEATURES).sort_values(ascending=True)

ax = feat_imp.plot(kind="barh", figsize=(8, 6))
ax.set_title("Mean |SHAP| — q_0.50")
ax.set_xlabel("Mean absolute SHAP value")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer
from scipy import stats as sp_stats

result = permutation_importance(
    models_last["q_0.50"],
    X_val_last,
    y_val_last,
    n_repeats=30,
    random_state=42,
    scoring=make_scorer(pinball_50),
)

pi_df = (
    pd.DataFrame({
        "feature": X_val_last.columns,
        "importance": result.importances_mean,
        "std": result.importances_std,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
pi_df.index += 1
pi_df["importance"] = pi_df["importance"].round(6)
pi_df["std"] = pi_df["std"].round(6)
pi_df["p_value"] = sp_stats.ttest_1samp(result.importances, 0.0, axis=1).pvalue.round(4)
pi_df["significant"] = pi_df["p_value"] < 0.05

print("Permutation importance (val set) — higher = more important, significant = p<0.05")
print(pi_df.to_string())


In [ ]:
train_mask_last = last_fold["train_mask"]
val_mask_last = last_fold["val_mask"]
X_tr_last = X[train_mask_last]
X_va_last = X[val_mask_last]
y_tr_last = y[train_mask_last]
y_va_last = y[val_mask_last]

ablation_df = run_feature_ablation(
    APM_FEATURES, X_tr_last, y_tr_last, X_va_last, y_va_last,
    xgb_params=XGB_PARAMS,
)


In [ ]:
from scipy import stats

model = models_last["q_0.50"]
_shap_expl = shap.TreeExplainer(model, model_output="raw")
_shap_vals = _shap_expl(X_val_last)
_mean_abs_shap = np.abs(_shap_vals.values).mean(axis=0)

shap_table = pd.DataFrame({
    "feature": X_val_last.columns,
    "mean_abs_shap": _mean_abs_shap,
}).assign(shap_rank=lambda d: d["mean_abs_shap"].rank(method="min", ascending=False).astype(int))

perm_table = (
    pi_df.sort_values("importance", ascending=False)
    .reset_index(drop=True)
    .rename(columns={"importance": "perm_importance", "std": "perm_importance_std"})
)
perm_table["perm_rank"] = np.arange(1, len(perm_table) + 1)
perm_table = perm_table[["feature", "perm_rank", "perm_importance", "perm_importance_std"]]

perm_pv = pd.DataFrame({
    "feature": X_val_last.columns,
    "perm_pvalue": stats.ttest_1samp(result.importances, 0.0, axis=1, nan_policy="omit").pvalue,
})

abla = ablation_df[["feature", "delta_rmse"]].rename(columns={"delta_rmse": "ablation_delta"})

feature_audit = (
    pd.DataFrame({"feature": APM_FEATURES})
    .merge(shap_table, on="feature", how="left")
    .merge(perm_table, on="feature", how="left")
    .merge(perm_pv, on="feature", how="left")
    .merge(abla, on="feature", how="left")
)

_ms = feature_audit["mean_abs_shap"].astype(float)
_mp = feature_audit["perm_importance"].astype(float)
_max_shap = _ms.fillna(0).max()
_max_perm = _mp.fillna(0).max()
feature_audit["shap_norm"] = np.where(_max_shap > 0, _ms.fillna(0) / _max_shap, 0.0)
feature_audit["perm_norm"] = np.where(_max_perm > 0, _mp.fillna(0) / _max_perm, 0.0)
feature_audit["consensus"] = (feature_audit["shap_norm"] + feature_audit["perm_norm"]) / 2
feature_audit = feature_audit.sort_values("consensus", ascending=False).reset_index(drop=True)
feature_audit.index += 1
feature_audit["keep"] = (feature_audit["perm_pvalue"] < 0.05) & (feature_audit["ablation_delta"] > 0)

print(feature_audit.to_string())


In [ ]:
correlated_features = analyze_correlations(
    df, APM_FEATURES, title="Feature Correlation Matrix — WNBA APM"
)
correlated_features


In [ ]:
holdout = evaluate_holdout(
    ppm_df,
    ppm_holdout,
    features=APM_FEATURES,
    target_col=TARGET_COL,
    xgb_params=XGB_PARAMS,
    role_col=ROLE_COL,
    tiers=APM_TIERS,
    wf_results=wf_results,
    quantiles=QUANTILES,
    fold_label=f"{HOLDOUT_SEASON} Blind Holdout",
)
models_ho = holdout["models_ho"]
preds_ho = holdout["preds_ho"]
ho_metrics = holdout["ho_metrics"]
X_ho = holdout["X_ho"]
y_ho = holdout["y_ho"]


In [ ]:
naive_holdout_results = evaluate_holdout_vs_naive(
    ppm_holdout,
    preds_ho,
    target_col=TARGET_COL,
    naive_primary=NAIVE_PRIMARY,
    naive_secondary=NAIVE_SECONDARY,
    naive_wf_primary=naive["naive_wf_primary"],
    wf_test_primary=naive["wf_test_primary"],
    alpha=ALPHA,
    holdout_label=f"{HOLDOUT_SEASON} holdout",
)


In [ ]:
from sklearn.metrics import mean_absolute_error

save_path = save_model_bundle(
    models_ho,
    train_df=ppm_df,
    holdout_df=ppm_holdout,
    wf_results=wf_results,
    ho_metrics=ho_metrics,
    features=APM_FEATURES,
    artifact_stem=ARTIFACT_STEM,
    naive_holdout_results=naive_holdout_results,
)

bundle = load_model_bundle(save_path)
models_loaded = bundle["quantile_models"]
feature_names = bundle["feature_names"]

test_preds = models_loaded["q_0.50"].predict(X_ho)
print(f"Loaded model MAE (holdout): {mean_absolute_error(y_ho, test_preds):.4f}")
print(f"Trained on data up to      : {bundle['train_end'].date()}")
print(f"Holdout through            : {bundle['val_end'].date()}")

predict_quantiles(models_loaded, feature_names, X_ho.head(3))
